# Early Sepsis Prediction: Part 3, Split Ablation

Compares **BASE** (basic preprocessing, leakage-free) with **NOVEL** (BASE + P1 to P6) under four splitting strategies, with XGBoost and logistic regression, for horizons of 6, 12, 24 and 48 hours:

1. S1 train / validation / test (70/15/15, 3 seeds)
2. S2 train / test (80/20, 3 seeds, 500 fixed rounds)
3. S3 5-fold cross-validation
4. S4 external: train on hospital A, test on B, and the other way round

Run notebook 02 first. Training saves one result per model, so after a disconnect just run the cells again.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL = "https://github.com/Thorfast191/Early-Sepsis-Detection-In-ICU-Patients.git"
REPO_DIR = "/content/Early-Sepsis-Detection-In-ICU-Patients"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

In [ ]:
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
FORCE = False       # set True to recompute steps instead of loading checkpoints
QUICK_MODE = False  # True: run on a stratified sample of 4,000 patients (own checkpoint folders)

RUN_V2 = False      # True: also train the v2 arms (data fixes, site screening, group ablation)

os.environ["SEPSIS_QUICK_MODE"] = "1" if QUICK_MODE else "0"
os.environ["SEPSIS_RUN_V2"] = "1" if RUN_V2 else "0"

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 60)

## GPU check

XGBoost uses the GPU when Colab has one (Runtime > Change runtime type > GPU).

In [ ]:
!nvidia-smi -L || echo "no GPU"

In [ ]:
import importlib
from data_preprocessing import config, utils
from novel_preprocessing import n07_fitted_transforms
from ablation import a01_make_splits, a02_train_eval, a03_stats, a04_plots

for module in [config, utils, n07_fitted_transforms, a01_make_splits, a02_train_eval, a03_stats, a04_plots]:
    importlib.reload(module)

print("quick mode:", config.QUICK_MODE)
print("XGBoost device:", a02_train_eval.xgb_device())

## 1. Patient-level splits

Splits are made once, by patient, stratified by (ever septic) x (hospital), and reused by both arms.

In [ ]:
splits = a01_make_splits.run(force=FORCE)

## 2. Train and evaluate

Every transform is fitted on the training patients of each split. Results go to `results/ablation_results.csv` (long format) and test predictions to `results/preds/`.

With `RUN_V2 = True` the v2 arms are trained too, on S1 seed 42 and both S4 directions (`config.V2_SPLITS`): the data fixes, the shared-support rule, label-free site screening (a hospital classifier per feature unit; units that identify the hospital with AUROC above 0.70 are dropped), leave-one-group-out arms and arms without ICULOS. Run notebook 02 step 8 first.

In [ ]:
results = a02_train_eval.run(force=FORCE)

## 3. Statistics

Bootstrap 95% CIs (1000 resamples of patients), paired NOVEL - BASE differences on the same test patients with DeLong p-values, and the generalisation gap (mean S1/S3 AUROC minus S4 AUROC).

In [ ]:
summary, paired, gaps = a03_stats.run(force=FORCE)

In [ ]:
utils.show(summary.loc["xgb"])

## 4. Plots

In [ ]:
a04_plots.run()

**Observations:** (write after running)